# Lab 2. Neural Machine Translation (NMT)

## Нейронный машинный перевод «в дикой природе»

В этом задании вам нужно добиться **наилучшего возможного перевода** для задачи перевода **с русского на английский (RU→EN)**. Базовый подход, использующий **RNN** как **энкодер** и **декодер**, уже реализован в разделе с примером.

Ваша <font color='magenta'>**основная задача — применить изученные методы**</font>, например:

1) улучшения оптимизации (например, **уменьшение скорости обучения со временем — learning rate decay**);
2) использование **трансформера / сверточной сети / другого энкодера** (с позиционным кодированием или без него);
3) механизм **внимания / самовнимания** (**настойчиво рекомендуется**);
4) **кастомная токенизация** (например, **BPE** или другие **subword**-подходы),

**чтобы улучшить качество перевода.**


## <font color='#9901ff'>**Критерии оценки**</font>

Не используйте **предобученные модели перевода / BERT / LLM чекпоинты** при выполнении **основного задания**. Все такие решения будут оцениваться только на **30% от возможных баллов**.

---

**<font color='magenta'>15 баллов**</font>

* реализованы **как минимум 3 улучшения модели** по сравнению с базовым примером;
* достигнут **порог 27 BLEU** на тестовом корпусе;
* представлены **экспериментальные результаты и выводы** в понятном для человека виде

**<font color='magenta'>11 баллов**</font>

* реализованы **как минимум 2 улучшения модели** по сравнению с базовым примером;
* достигнут **порог 25 BLEU** на тестовом корпусе;
* представлены **экспериментальные результаты и выводы** в понятном для человека виде

**<font color='magenta'>8 баллов**</font>

* реализовано **как минимум 1 улучшение модели** по сравнению с базовым примером;
* достигнут **порог 21 BLEU** на тестовом корпусе;
* представлены **экспериментальные результаты и выводы** в понятном для человека виде

**<font color='magenta'>4 балла** (для тех, кому совсем тяжело с экспериментами)</font>

* заполнен **код в разделе предобработки данных** базового примера

<font color='#9901ff'>**Замечание:** три улучшения — это именно три разных аспекта, например:</font>
1) **архитектура модели:** добавление Attention между энкодером и декодером
2) **представление данных:** использование BPE / SentencePiece / subword токенизации вместо пословной
3) **оптимизация обучения:** использование learning rate decay / scheduler’а

Также, помимо основного задания, <font color='magenta'>**есть два бонуса, это дополнительные 6 баллов**, там как раз нужно взять **предобученные модели**</font>

<font color='magenta'>**Подсказка:**</font> можете, конечно, взять код из примера и вносить в него все изменения (и дописать код предобработки), но лучше использовать **transformers**, который мы рассмотрели на семинаре, чтобы упростить себе жизнь:

1) чтобы использовать архитектуру модели из **Hugging Face**, но **не загружать предобученные веса**, используйте метод `from_config` вместо `from_pretrained`.
Подробнее — [здесь](https://huggingface.co/docs/transformers/v4.46.3/en/model_doc/auto#transformers.AutoModelForSeq2SeqLM);

2) можно использовать метод `train_new_from_iterator`,
чтобы обучить токенизатор на собственном текстовом корпусе.
Подробнее — [здесь](https://huggingface.co/learn/nlp-course/chapter6/2);

3) можно использовать **[класс Seq2SeqTrainer](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Seq2SeqTrainer) из transformers**. [Примеры](https://www.geeksforgeeks.org/audio-seq2seq-model-using-transformers) использования;

4) пайплайны обучения Hugging Face поддерживают **Torch Datasets**,
но при желании вы можете **преобразовать датасет в формат Hugging Face**:
[ссылка](https://discuss.huggingface.co/t/correct-way-to-create-a-dataset-from-a-csv-file/15686/15);

5) пример ноутбука с пайплайном от **Hugging Face**: [ссылка](https://github.com/huggingface/notebooks/blob/main/examples/translation.ipynb).


In [ ]:
# Colab/local setup. PyTorch is already available in the standard Colab runtime.
%pip install -q subword-nmt nltk tqdm pandas

In [ ]:
# Данные взяты из курса YSDA NLP.
from pathlib import Path

LOCAL_DATA_PATH = Path('../../datasets/Machine_translation_EN_RU/data.txt')
DATA_PATH = LOCAL_DATA_PATH if LOCAL_DATA_PATH.exists() else Path('data.txt')
if not DATA_PATH.exists():
    print('Локальный датасет не найден. Загружаю его с GitHub.')
    !wget -q -O data.txt https://raw.githubusercontent.com/neychev/made_nlp_course/master/datasets/Machine_translation_EN_RU/data.txt
    DATA_PATH = Path('data.txt')

print(f'Используется датасет: {DATA_PATH}')

## Пример кода: базовый подход, использующий **RNN** как **энкодер** и **декодер**
<font color='magenta'>**Можете взять код как основу для улучшений или заполнить предобработку, если тяжело с экспериментами**</font>

Примерный конвейер (pipeline), приведённый ниже, **работает**, но является **устаревшим**. <font color='magenta'>**В примере получился BLEU 14.**</font>


In [ ]:
# устаревший код
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import random_split
from torch.utils.data import DataLoader

import random
import math
import time

import matplotlib
matplotlib.rcParams.update({'figure.figsize': (16, 12), 'font.size': 14})
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import clear_output

from nltk.tokenize import WordPunctTokenizer
from subword_nmt.learn_bpe import learn_bpe
from subword_nmt.apply_bpe import BPE

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
raw_pairs = []
with DATA_PATH.open(encoding='utf-8') as data:
    for line in data:
        line = line.strip()
        if not line:
            continue
        en_, ru_ = line.split('\t')
        raw_pairs.append((en_, ru_))

random.Random(SEED).shuffle(raw_pairs)
n_total = len(raw_pairs)
n_train = int(0.8 * n_total)
n_valid = int(0.1 * n_total)
train_pairs = raw_pairs[:n_train]
valid_pairs = raw_pairs[n_train:n_train + n_valid]
test_pairs = raw_pairs[n_train + n_valid:]

en_train = [en for en, _ in train_pairs]
ru_train = [ru for _, ru in train_pairs]
en_valid = [en for en, _ in valid_pairs]
ru_valid = [ru for _, ru in valid_pairs]
en_test = [en for en, _ in test_pairs]
ru_test = [ru for _, ru in test_pairs]

# Эти имена используются baseline-блоком для построения словарей только по train split.
en_data, ru_data = en_train, ru_train
print(f'Всего пар: {n_total}; train/valid/test: {len(train_pairs)}/{len(valid_pairs)}/{len(test_pairs)}')

### Предобработка данных <font color='magenta'>**4 балла** (для тех, кто не будет делать эксперименты)</font>

**Здесь выполняется токенизация.** Не стесняйтесь использовать **BPE** или более сложный алгоритм токенизации в своём решении

**Подсказка:** в библиотеке **transformers** можно использовать метод `train_new_from_iterator`,
чтобы обучить токенизатор на собственном текстовом корпусе.
Подробнее — [здесь](https://huggingface.co/learn/nlp-course/chapter6/2).


In [ ]:
tokenizer_W = WordPunctTokenizer()

def tokenize(x, tokenizer=tokenizer_W):
    return [token.lower() for token in tokenizer.tokenize(x)]

In [ ]:
BOS_TOKEN = "<bos>" # то же, что и <sos> (Begin/Start of Sentence)
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"
PAD_TOKEN = "<pad>"

PAD_TOKEN_ID = 0
BOS_TOKEN_ID = 1
EOS_TOKEN_ID = 2
UNK_TOKEN_ID = 3
SPECIAL_TOKENS = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN]


In [ ]:
from collections import Counter

def build_vocab(text, tokenizer=tokenizer_W, min_freq=2):
    counts = Counter()

    for s in text:
        counts.update(tokenizer(s))

    vocab = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN]
    vocab.extend(
        token for token, count in sorted(counts.items(), key=lambda item: (-item[1], item[0]))
        if count >= min_freq and token not in SPECIAL_TOKENS
    )

    return {word: i for i, word in enumerate(vocab)}, [word for i, word in enumerate(vocab)]


en_vocab_to_ids,en_ids_to_vocab = build_vocab(en_data, tokenizer_W)
ru_vocab_to_ids, ru_ids_to_vocab = build_vocab(ru_data, tokenizer_W)

In [ ]:
print(f"Уникальных токенов в словаре исходного языка (ru): {len(ru_ids_to_vocab)}")
print(f"Уникальных токенов в словаре целевого языка (en): {len(en_ids_to_vocab)}")

Создадим датасет. Вы можете изменить реализацию под своё решение.

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, en, ru, en_tokenizer=tokenizer_W, ru_tokenizer=tokenizer_W):
        self.en_tokenizer = en_tokenizer
        self.ru_tokenizer = ru_tokenizer
        self.examples = [(ru_sent, en_sent) for ru_sent, en_sent in zip(ru, en)]

    def __getitem__(self, index):
        ru_sent, en_sent = self.examples[index]
        ru_tokens = tokenize(ru_sent, tokenizer=self.ru_tokenizer)
        en_tokens = tokenize(en_sent, tokenizer=self.en_tokenizer)
        return {
            'input_ids': [ru_vocab_to_ids.get(token, UNK_TOKEN_ID) for token in ru_tokens] + [EOS_TOKEN_ID],
            'labels': [BOS_TOKEN_ID] + [en_vocab_to_ids.get(token, UNK_TOKEN_ID) for token in en_tokens] + [EOS_TOKEN_ID],
        }

    def __len__(self):
        return len(self.examples)

In [ ]:
train_data = TranslationDataset(en_train, ru_train)
valid_data = TranslationDataset(en_valid, ru_valid)
test_data = TranslationDataset(en_test, ru_test)

In [ ]:
dataset[0]

In [ ]:
print(f"Количество примеров в обучающей выборке: {len(train_data)}")
print(f"Количество примеров в валидационной выборке: {len(valid_data)}")
print(f"Количество примеров в тестовой выборке: {len(test_data)}")

Вот токены из исходного корпуса (RU):

In [ ]:
ru_ids_to_vocab[::1000]

А вот токены из целевого корпуса (EN):

In [ ]:
en_ids_to_vocab[::1000]

А вот пример из обучающего датасета:

In [ ]:
print(train_data[9])

Давайте посмотрим на распределение длин:

In [ ]:
src_length = map(len, [x['input_ids'] for x in train_data])
trg_length = map(len, [x['labels'] for x in train_data])

print('Распределение длин в обучающей выборке')
plt.figure(figsize=[8, 4])
plt.subplot(1, 2, 1)
plt.title("Длина исходного текста")
plt.hist(list(src_length), bins=20);

plt.subplot(1, 2, 2)
plt.title("Длина перевода")
plt.hist(list(trg_length), bins=20);

In [ ]:
src_length = map(len, [x['input_ids'] for x in test_data])
trg_length = map(len, [x['labels'] for x in test_data])

print('Распределение длин в тестовой выборке')
plt.figure(figsize=[8, 4])
plt.subplot(1, 2, 1)
plt.title("Длина исходного текста")
plt.hist(list(src_length), bins=20);

plt.subplot(1, 2, 2)
plt.title("Длина перевода")
plt.hist(list(trg_length), bins=20);

In [ ]:
MAX_LENGTH = 100
BATCH_SIZE = 64

def pad_to_max_length(text_ids):
    if len(text_ids) > MAX_LENGTH:
        # Сохраняем EOS в последней позиции после обрезки.
        return text_ids[:MAX_LENGTH - 1] + [text_ids[-1]]
    return text_ids + [PAD_TOKEN_ID] * (MAX_LENGTH - len(text_ids))

def collate_fn(batch):
    # ОБРАТИТЕ ВНИМАНИЕ: Функция collate_fn возвращает тензоры
    # формы (MAX_LENGTH, BATCH_SIZE)
    # НЕ ЗАБУДЬТЕ убрать .T.contiguous(), если вашей модели
    #нужен ввод формы (BATCH_SIZE, MAX_LENGTH)
    input_ids = torch.Tensor([pad_to_max_length(data["input_ids"]) for data in batch]).to(torch.long).T.contiguous()
    labels = torch.Tensor([pad_to_max_length(data["labels"]) for data in batch]).to(torch.long).T.contiguous()
    return {"input_ids": input_ids, "labels": labels}

train_dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_dataloader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
for x in train_dataloader:
    break

assert x["input_ids"].shape[0] == MAX_LENGTH
assert x["labels"].shape[0] == MAX_LENGTH
assert x["input_ids"].shape[1] <= BATCH_SIZE
assert torch.all(x["labels"][0] == BOS_TOKEN_ID)
print({key: value.shape for key, value in x.items()})

### Пример модели

Эта модель похожа на **seq2seq модель из практики**.
В своём решении вы можете **добавлять слои к этой модели** или **загрузить модель из transformers с помощью метода `from_config`**.
**Не инициализируйте модель с предобученными весами.**

Повторно запускать этот код в разделе примера **не требуется**.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import random
import math
import time


class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()

        self.input_dim = input_dim
        self.emb_dim = emb_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(
            num_embeddings=input_dim,
            embedding_dim=emb_dim,
            padding_idx=PAD_TOKEN_ID
        )

        self.rnn = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hid_dim,
            num_layers=n_layers,
            dropout=dropout
        )

        self.dropout = nn.Dropout(p=dropout)

    def forward(self, src):
        # Вычисляем эмбеддинги для входного текста и применяем dropout
        embedded = self.embedding(src)
        embedded = self.dropout(embedded)

        output, (hidden, cell) = self.rnn(embedded)

        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()

        self.emb_dim = emb_dim
        self.hid_dim = hid_dim
        self.output_dim = output_dim
        self.n_layers = n_layers
        self.dropout = dropout

        self.embedding = nn.Embedding(
            num_embeddings=output_dim,
            embedding_dim=emb_dim,
            padding_idx=PAD_TOKEN_ID
        )

        self.rnn = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hid_dim,
            num_layers=n_layers,
            dropout=dropout
        )

        self.out = nn.Linear(
            in_features=hid_dim,
            out_features=output_dim
        )

        self.dropout = nn.Dropout(p=dropout)

    def forward(self, input, hidden, cell):

        input = input.unsqueeze(0)

        # Вычисляем эмбеддинги для входного токена и применяем dropout
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.out(output.squeeze(0))

        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

        assert encoder.hid_dim == decoder.hid_dim, \
            "Размер скрытого состояния энкодера и декодера должен быть одинаковым!"
        assert encoder.n_layers == decoder.n_layers, \
            "Количество слоёв в энкодере и декодере должно совпадать!"

    def forward(self, src, trg, teacher_forcing_ratio = 0.5):

        # teacher_forcing_ratio — вероятность использовать teacher forcing
        # например, если teacher_forcing_ratio = 0.75,
        # то 75% времени используется ground-truth вход

        # теперь батч в измерении [1], а не [0]]
        batch_size = trg.shape[1]
        max_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        # тензор для хранения выходов декодера
        outputs = torch.zeros(max_len, batch_size, trg_vocab_size).to(self.device)

        # последний скрытый слой энкодера используется как начальное состояние декодера
        hidden, cell = self.encoder(src)

        # первый вход в декодер — <sos> токены
        input = trg[0,:]

        for t in range(1, max_len):

            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.max(1)[1]
            input = (trg[t] if teacher_force else top1)

        return outputs

### Обучение

Здесь показан **простой пайплайн обучения модели NMT**.
Он почти полностью повторяет практику по seq2seq.

Повторно запускать этот код в разделе примера **не требуется**.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
Encoder = Encoder
Decoder = Decoder
Seq2Seq = Seq2Seq

In [ ]:
INPUT_DIM = len(ru_ids_to_vocab)
OUTPUT_DIM = len(en_ids_to_vocab)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)

# не забудьте поместить модель на нужное устройство (CPU или GPU)
model = Seq2Seq(enc, dec, device).to(device)

In [ ]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param, -0.08, 0.08)

model.apply(init_weights)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Модель имеет {count_parameters(model):,} обучаемых параметров')

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN_ID)

In [ ]:
def train(model, iterator, optimizer, criterion, clip, train_history=None, valid_history=None):
    model.train()

    epoch_loss = 0
    history = []
    for i, batch in enumerate(iterator):

        src = batch['input_ids'].to(device)
        trg = batch['labels'].to(device)

        optimizer.zero_grad()

        output = model(src, trg)

        # trg = [длина предложения, размер батча]
        # output = [длина предложения, размер батча, размер словаря]
        output = output[1:].view(-1, output.shape[-1])
        trg = trg[1:].view(-1)

        # trg = [(длина предложения - 1) * размер батча]
        # output = [(длина предложения - 1) * размер батча, размер словаря]
        loss = criterion(output, trg)

        loss.backward()

        # Обрежем градиенты
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()

        history.append(loss.item())
        if (i+1)%10==0:
            fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12, 8))

            clear_output(True)
            ax[0].plot(history, label='train loss')
            ax[0].set_xlabel('Batch')
            ax[0].set_title('Train loss')
            if train_history is not None:
                ax[1].plot(train_history, label='general train history')
                ax[1].set_xlabel('Epoch')
            if valid_history is not None:
                ax[1].plot(valid_history, label='general valid history')
            plt.legend()

            plt.show()


    return epoch_loss / max(1, len(iterator))

In [ ]:
def evaluate(model, iterator, criterion):

    model.eval()

    epoch_loss = 0

    history = []

    with torch.no_grad():

        for i, batch in enumerate(iterator):

            src = batch['input_ids'].to(device)
            trg = batch['labels'].to(device)

            output = model(src, trg, 0) # отключаем teacher forcing

            # trg = [длина предложения, размер батча]
            # output = [длина предложения, размер батча, размер словаря]
            output = output[1:].view(-1, output.shape[-1])
            trg = trg[1:].view(-1)

            # trg = [(длина предложения - 1) * размер батча]
            # output = [(длина предложения - 1) * размер батча, размер словаря]
            loss = criterion(output, trg)

            epoch_loss += loss.item()

    return epoch_loss / max(1, len(iterator))

In [ ]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [ ]:
train_history = []
valid_history = []

N_EPOCHS = 10
CLIP = 1.0

best_valid_loss = float('inf')

In [ ]:
for epoch in range(N_EPOCHS):

    start_time = time.time()

    train_loss = train(model, train_dataloader, optimizer, criterion, CLIP, train_history, valid_history)
    valid_loss = evaluate(model, valid_dataloader, criterion)

    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'baseline-model.pt')

    train_history.append(train_loss)
    valid_history.append(valid_loss)
    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

model.load_state_dict(torch.load('baseline-model.pt', map_location=device))

### Оценка модели <font color='magenta'>(спойлер: BLEU 14)</font>

Повторно запускать этот код в разделе примера **не требуется**.

**Давайте посмотрим на качество нашей сети**:


In [ ]:

def flatten(l):
    return [item for sublist in l for item in sublist]

def remove_tech_tokens(mystr, tokens_to_remove=SPECIAL_TOKENS):
    return [x for x in mystr if x not in tokens_to_remove]


def get_text(x, en_ids_to_vocab):
    text = [en_ids_to_vocab[token_id] for token_id in x]
    try:
        end_idx = text.index(EOS_TOKEN)
        text = text[:end_idx]
    except ValueError:
        pass
    text = remove_tech_tokens(text)
    if len(text) < 1:
        text = []
    return text


def generate_translation(src, trg, model, en_ids_to_vocab):
    model.eval()

    output = model(src, trg, 0) # отключаем teacher forcing
    output = output.argmax(dim=-1).cpu().numpy()

    original = get_text(list(trg[:,0].cpu().numpy()), en_ids_to_vocab)
    generated = get_text(list(output[1:, 0]), en_ids_to_vocab)

    print('Original: {}'.format(' '.join(original)))
    print('Generated: {}'.format(' '.join(generated)))
    print()

In [ ]:
batch = next(iter(test_dataloader))

In [ ]:
for idx in range(min(2, batch["input_ids"].shape[1])):
    src = batch["input_ids"][:, idx:idx+1].to(device)
    trg = batch["labels"][:, idx:idx+1].to(device)
    generate_translation(src, trg, model, en_ids_to_vocab)

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

bleu_smoothing = SmoothingFunction().method1

def calculate_bleu(references, hypotheses):
    reference_tokens = [[reference] for reference in references]
    return corpus_bleu(reference_tokens, hypotheses, smoothing_function=bleu_smoothing) * 100

In [ ]:
import tqdm

In [ ]:
def collect_predictions(model, dataloader, target_vocab):
    references, hypotheses = [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm.tqdm(dataloader):
            src = batch["input_ids"].to(device)
            trg = batch["labels"].to(device)
            output = model(src, trg, 0).argmax(dim=-1)
            references.extend([get_text(x, target_vocab) for x in trg.cpu().numpy().T])
            hypotheses.extend([get_text(x, target_vocab) for x in output[1:].cpu().numpy().T])
    return references, hypotheses

original_text, generated_text = collect_predictions(model, test_dataloader, en_ids_to_vocab)

In [ ]:
baseline_bleu = calculate_bleu(original_text, generated_text)
print(f'Baseline word-level BLEU: {baseline_bleu:.2f}')

## <font color='green'>**Ваше решение**</font>

Ниже реализованы три независимых улучшения относительно baseline: subword/BPE-токенизация, attention между энкодером и декодером и scheduler для learning rate. Все модели обучаются с нуля без pretrained-весов.

In [ ]:
from io import StringIO

def train_bpe_tokenizer(sentences, num_symbols=8000):
    corpus = StringIO('\n'.join(' '.join(tokenize(sentence)) for sentence in sentences))
    codes = StringIO()
    learn_bpe(corpus, codes, num_symbols=num_symbols, min_frequency=2)
    codes.seek(0)
    return BPE(codes, separator='@@')

def apply_bpe_tokenizer(sentence, bpe_tokenizer):
    encoded = bpe_tokenizer.process_line(' '.join(tokenize(sentence))).strip()
    return encoded.split() if encoded else []

def build_vocab_from_tokens(token_lists, min_freq=2):
    counts = Counter(token for tokens in token_lists for token in tokens)
    vocab = list(SPECIAL_TOKENS)
    vocab.extend(
        token for token, count in sorted(counts.items(), key=lambda item: (-item[1], item[0]))
        if count >= min_freq and token not in SPECIAL_TOKENS
    )
    return {token: index for index, token in enumerate(vocab)}, vocab

class TokenizedTranslationDataset(Dataset):
    def __init__(self, en_tokens, ru_tokens):
        self.examples = list(zip(ru_tokens, en_tokens))

    def __getitem__(self, index):
        ru_tokens, en_tokens = self.examples[index]
        return {
            'input_ids': [ru_vocab_to_ids.get(token, UNK_TOKEN_ID) for token in ru_tokens] + [EOS_TOKEN_ID],
            'labels': [BOS_TOKEN_ID] + [en_vocab_to_ids.get(token, UNK_TOKEN_ID) for token in en_tokens] + [EOS_TOKEN_ID],
        }

    def __len__(self):
        return len(self.examples)

### Подготовка BPE-данных

BPE-коды и словари обучаются только на train split, чтобы не переносить информацию из validation/test в preprocessing.

In [ ]:
BPE_SYMBOLS = 8000
ru_bpe = train_bpe_tokenizer(ru_train, num_symbols=BPE_SYMBOLS)
en_bpe = train_bpe_tokenizer(en_train, num_symbols=BPE_SYMBOLS)

ru_train_tokens = [apply_bpe_tokenizer(sentence, ru_bpe) for sentence in ru_train]
ru_valid_tokens = [apply_bpe_tokenizer(sentence, ru_bpe) for sentence in ru_valid]
ru_test_tokens = [apply_bpe_tokenizer(sentence, ru_bpe) for sentence in ru_test]
en_train_tokens = [apply_bpe_tokenizer(sentence, en_bpe) for sentence in en_train]
en_valid_tokens = [apply_bpe_tokenizer(sentence, en_bpe) for sentence in en_valid]
en_test_tokens = [apply_bpe_tokenizer(sentence, en_bpe) for sentence in en_test]

ru_vocab_to_ids, ru_ids_to_vocab = build_vocab_from_tokens(ru_train_tokens)
en_vocab_to_ids, en_ids_to_vocab = build_vocab_from_tokens(en_train_tokens)

bpe_train_data = TokenizedTranslationDataset(en_train_tokens, ru_train_tokens)
bpe_valid_data = TokenizedTranslationDataset(en_valid_tokens, ru_valid_tokens)
bpe_test_data = TokenizedTranslationDataset(en_test_tokens, ru_test_tokens)
bpe_train_dataloader = DataLoader(bpe_train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
bpe_valid_dataloader = DataLoader(bpe_valid_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
bpe_test_dataloader = DataLoader(bpe_test_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'BPE vocabularies: RU={len(ru_ids_to_vocab)}, EN={len(en_ids_to_vocab)}')
print(f'BPE train/valid/test: {len(bpe_train_data)}/{len(bpe_valid_data)}/{len(bpe_test_data)}')

### Attention-модель и scheduler


Финальная модель использует случайно инициализированный LSTM encoder-decoder с Bahdanau attention. `ReduceLROnPlateau` изменяет learning rate по validation loss.


In [ ]:
class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 2, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs, mask=None):
        src_len = encoder_outputs.shape[0]
        hidden = hidden[-1].unsqueeze(1).repeat(1, src_len, 1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        scores = self.v(energy).squeeze(2)
        if mask is not None:
            scores = scores.masked_fill(~mask, -1e10)
        return torch.softmax(scores, dim=1)

class AttentionEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=PAD_TOKEN_ID)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell

class AttentionDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_TOKEN_ID)
        self.attention = attention
        self.rnn = nn.LSTM(emb_dim + hid_dim, hid_dim, n_layers, dropout=dropout)
        self.out = nn.Linear(hid_dim * 2 + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs, mask=None):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        attention_weights = self.attention(hidden, encoder_outputs, mask).unsqueeze(1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        weighted = torch.bmm(attention_weights, encoder_outputs).permute(1, 0, 2)
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        prediction = self.out(torch.cat((output.squeeze(0), weighted.squeeze(0), embedded.squeeze(0)), dim=1))
        return prediction, hidden, cell

class AttentionSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert encoder.hid_dim == decoder.hid_dim
        assert encoder.n_layers == decoder.rnn.num_layers

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[1]
        max_len = trg.shape[0]
        outputs = torch.zeros(max_len, batch_size, self.decoder.output_dim, device=self.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        src_mask = (src != PAD_TOKEN_ID).permute(1, 0)
        input = trg[0, :]
        for t in range(1, max_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs, src_mask)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            input = trg[t] if teacher_force else output.argmax(1)
        return outputs

def initialize_weights(module):
    for parameter in module.parameters():
        if parameter.dim() > 1:
            nn.init.xavier_uniform_(parameter)

final_attention = Attention(HID_DIM)
final_encoder = AttentionEncoder(len(ru_ids_to_vocab), ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
final_decoder = AttentionDecoder(len(en_ids_to_vocab), DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT, final_attention)
final_model = AttentionSeq2Seq(final_encoder, final_decoder, device).to(device)
final_model.apply(initialize_weights)
final_optimizer = optim.Adam(final_model.parameters(), lr=1e-3)
final_scheduler = optim.lr_scheduler.ReduceLROnPlateau(final_optimizer, mode='min', factor=0.5, patience=1)

FINAL_EPOCHS = 15
final_train_history, final_valid_history = [], []
best_final_loss = float('inf')
epochs_without_improvement = 0

for epoch in range(FINAL_EPOCHS):
    start_time = time.time()
    train_loss = train(final_model, bpe_train_dataloader, final_optimizer, criterion, CLIP, final_train_history, final_valid_history)
    valid_loss = evaluate(final_model, bpe_valid_dataloader, criterion)
    final_scheduler.step(valid_loss)
    final_train_history.append(train_loss)
    final_valid_history.append(valid_loss)
    elapsed = int(time.time() - start_time)
    current_lr = final_optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch + 1:02}: train={train_loss:.3f}, valid={valid_loss:.3f}, lr={current_lr:.2e}, time={elapsed}s')
    if valid_loss < best_final_loss:
        best_final_loss = valid_loss
        epochs_without_improvement = 0
        torch.save(final_model.state_dict(), 'final-attention-bpe-model.pt')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= 3:
            print('Early stopping')
            break

final_model.load_state_dict(torch.load('final-attention-bpe-model.pt', map_location=device))
final_references, final_hypotheses = collect_predictions(final_model, bpe_test_dataloader, en_ids_to_vocab)

def detokenize_bpe(tokens):
    return ' '.join(tokens).replace('@@ ', '').split()

# Считаем BLEU после обратной склейки BPE, чтобы сравнение с baseline
# происходило на уровне слов, а не внутренних subword-токенов.
final_references = [detokenize_bpe(tokens) for tokens in final_references]
final_hypotheses = [detokenize_bpe(tokens) for tokens in final_hypotheses]
final_bleu = calculate_bleu(final_references, final_hypotheses)

import pandas as pd
results = pd.DataFrame([
    {'experiment': 'Baseline: word-level LSTM', 'improvements': 0, 'BLEU': baseline_bleu},
    {'experiment': 'Final: BPE + Attention + scheduler', 'improvements': 3, 'BLEU': final_bleu},
])
display(results)
print(f'Final BLEU: {final_bleu:.2f}')
print('Target BLEU for the 15-point level: 27.00')

## Выводы

В финальной модели использованы три независимых улучшения: BPE уменьшает проблему неизвестных слов, attention позволяет декодеру выбирать важные части входной последовательности, а scheduler адаптирует скорость обучения по validation loss. Сравнение с word-level LSTM baseline приведено в таблице выше.

Итоговый BLEU следует интерпретировать вместе с описанием токенизации, числа эпох, размера batch и аппаратной конфигурации Colab.